# 2.1.5 One-layer versus two-layer LSTM experiment

Train a fresh one-layer control and a fresh two-layer LSTM with target-aware attention on an A100 / High RAM Colab runtime. The second layer processes the first layer's sequence of hidden states; attention uses the top layer's states. Both layers have 128 hidden units. No new architecture helper is needed.

**Question:** Does stacking another LSTM layer improve news retrieval enough to justify additional parameters and inference cost?

Upload the existing `lstm_helpers.zip` to `/content`, and add the same shared Drive shortcut used in 2.1.3. The completed initial run supplies hyperparameters. Run all cells in order. Outputs go to a separate `2.1.5_depth_experiment/<run_id>/` folder and do not overwrite prior runs.

Default: train depths 1 and 2 with seed 42. Set `SEEDS = (42, 43, 44)` before running for matched multi-seed comparisons. Both receive eight complete training epochs and the same initial learning rate, with a constant learning rate and no early stopping, to match training exposure. Checkpoint selection still uses validation. This differs from 2.1.3's adaptive learning-rate/early-stopping procedure, so its published results are context, not a matched control.

**Scope:** training and validation only, no project-test request loading or scoring, no exhaustive prediction exports. Selection uses 2,000 validation requests; reporting uses 5,000 different validation requests. Both remain development data. Do not select the best seed using reporting results.

**Depth and dropout:** the existing helper enables inter-layer dropout only when depth exceeds one. By default this experiment tests the standard two-layer configuration including that dropout. Set `INTER_LAYER_DROPOUT = 0.0` to disable it in the stacked model while leaving other dropout unchanged. Record this choice when interpreting the comparison.

In [ ]:
from pathlib import Path
EXPERIMENT_ID = "2.1_lstm"
EXPERIMENT_STAGE = "2.1.5"
DATA_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder")
REFERENCE_RUN_ID = "20261104_014914_seed42"
SEED = 42  # Fixed sampling seed, independent of model seed.
SEEDS = (42,)  # Change to (42, 43, 44) for the multi-seed comparison.
DEPTHS = (1, 2)
MAX_EPOCHS = 8
INTER_LAYER_DROPOUT = None  # None = existing configured dropout; 0.0 = disable.
HISTORY_MAX_LEN = 50
SELECTION_REQUESTS, REPORT_REQUESTS = 2000, 5000
SELECTION_K, EXPORT_K = 100, 200
FRESHNESS_GRID = (0., 0.25, 0.5, 1., 2., 4., 8., 16., 64.)
# "blended" follows the combined-score objective; "raw" isolates raw neural retrieval.
CHECKPOINT_OBJECTIVE = "blended"
PAIR_CHUNK, REQUEST_BATCH = 131072, 16
RELOAD_TOLERANCE = 1e-4
SELECTION_OBJECTIVE = "known_click_recall@100"
SELECTION_TIEBREAK = "observed_click_ndcg@100"
ACTIVE_SPLITS = ("train", "validation")
assert CHECKPOINT_OBJECTIVE in ("raw", "blended")
assert DEPTHS == (1, 2), "Keep both depths for a matched comparison."
assert len(SEEDS) == len(set(SEEDS)) and SEEDS
assert MAX_EPOCHS > 0
assert INTER_LAYER_DROPOUT is None or 0 <= INTER_LAYER_DROPOUT < 1


## 1. Runtime, helpers and reference hyperparameters

In [ ]:
import os, sys, subprocess, platform, json, shutil, time, hashlib, importlib.util, sqlite3
from importlib import metadata
from dataclasses import fields, asdict

if "google.colab" in sys.modules or Path("/content").is_dir():
    from google.colab import drive
    drive.mount("/content/drive")
    CODE_DIR = Path("/content/lstm_depth_code")
    CODE_DIR.mkdir(exist_ok=True)
    import zipfile
    with zipfile.ZipFile("/content/lstm_helpers.zip") as archive:
        archive.extractall(CODE_DIR)
else:
    CODE_DIR = Path.cwd()
for name in ("lstm_data.py", "lstm_model.py", "lstm_inference.py"):
    if not (CODE_DIR / name).is_file():
        raise FileNotFoundError(f"Place {name} in {CODE_DIR} before continuing.")

# Keep the runtime's core stack; install only the small missing dependencies.
for dist in ("duckdb", "optuna"):
    try:
        metadata.version(dist)
    except metadata.PackageNotFoundError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", dist], check=True)
sys.path.insert(0, str(CODE_DIR))
for name in ("lstm_data", "lstm_model", "lstm_inference"):
    sys.modules.pop(name, None)
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import optuna
from IPython.display import display
import lstm_data as D
import lstm_model as M
import lstm_inference as F

assert D.EXPORT_SCHEMA_VERSION == 4, "Upload the current schema-v4 helpers."
assert Path(D.__file__).resolve().parent == CODE_DIR.resolve()
assert Path(M.__file__).resolve().parent == CODE_DIR.resolve()
required_paths = set(D.PATH_CANDIDATES) - D.OPTIONAL_PATHS
required_paths -= {"features_test", "history_test"}
PATHS, PATH_NOTES = D.resolve_paths(DATA_ROOT, required=required_paths)
SPLIT_MANIFEST = json.loads((DATA_ROOT / "split_manifest.json").read_text())
if SPLIT_MANIFEST.get("protocol") != "ebnerd_small_labelled_user_holdout_v1":
    raise ValueError("Expected the labelled project dataset and its split manifest.")
if json.loads((DATA_ROOT / "feature_engineering" / "split_manifest.json").read_text()) != SPLIT_MANIFEST:
    raise ValueError("Feature outputs and dataset split manifest do not match.")

RUN_ID = pd.Timestamp.now(tz="Asia/Singapore").strftime("%Y%m%d_%H%M%S") + "_depth_comparison"
LOCAL_ROOT = Path("/content/lstm_outputs") if Path("/content").is_dir() else CODE_DIR / "outputs"
RUN_DIR = LOCAL_ROOT / RUN_ID
PERSIST_RUN_DIR = DATA_ROOT / "Experiments" / EXPERIMENT_ID / "2.1.5_depth_experiment" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
D.write_json(SPLIT_MANIFEST, RUN_DIR / "split_manifest.json")
PERSIST_RUN_DIR.mkdir(parents=True, exist_ok=False)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
M.set_seed(SEED)
VERSIONS = {dist: metadata.version(dist) for dist in
            ("torch", "numpy", "pandas", "pyarrow", "duckdb", "optuna")}
INPUTS = D.describe_inputs(PATHS, hash_files=False)  # path/size/mtime, not content hashes

def log(*parts):
    print(*parts, flush=True)

def copy_tree(source, destination):
    # Copy completed files atomically. The live Optuna database is outside RUN_DIR.
    for source_file in sorted(Path(source).rglob("*")):
        if source_file.is_file():
            D._copy_atomic(source_file, Path(destination) / source_file.relative_to(source))

log("Run", RUN_ID, "on", DEVICE, "| persistent folder:", PERSIST_RUN_DIR)
log("Dependencies:", VERSIONS)
log("GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU; select an A100 runtime")
REFERENCE_DIR = DATA_ROOT / "Experiments" / EXPERIMENT_ID / "2.1.1_initial_run" / REFERENCE_RUN_ID
if not REFERENCE_DIR.is_dir():
    REFERENCE_DIR = DATA_ROOT / "Experiments" / EXPERIMENT_ID / REFERENCE_RUN_ID
REFERENCE_CONFIG = json.loads((REFERENCE_DIR / "run_config.json").read_text())
REFERENCE_TUNING = json.loads((REFERENCE_DIR / "tuning" / "summary.json").read_text())
MODEL_KEYS = ("article_dim", "category_dim", "type_dim", "topic_dim", "text_dim", "image_dim",
              "lstm_hidden", "lstm_layers", "attention_dim", "interaction_dim", "head_hidden",
              "dropout", "embedding_dropout", "use_id_embedding", "id_dim", "text_dropout")
MODEL_HPARAMS = {k: REFERENCE_CONFIG["model"][k] for k in MODEL_KEYS}
TRAIN_HPARAMS = {k: v for k, v in REFERENCE_CONFIG["train_config"].items() if k != "seed"}
D.write_json({"reference_run_id": REFERENCE_RUN_ID, "neural_tuning": REFERENCE_TUNING,
              "model_hparams": MODEL_HPARAMS, "train_hparams": TRAIN_HPARAMS},
             RUN_DIR / "reference_hyperparameters.json")
if DEVICE.type != "cuda":
    raise RuntimeError("Choose a GPU runtime (A100 / High RAM recommended) before fitting.")
MODEL_HPARAMS.update(lstm_hidden=128, text_dropout=0.2)
D.write_json({"seeds": list(SEEDS), "depths": list(DEPTHS), "max_epochs": MAX_EPOCHS,
    "checkpoint_objective": CHECKPOINT_OBJECTIVE,
    "inter_layer_dropout_override": INTER_LAYER_DROPOUT,
    "training_schedule": "constant learning rate; fixed epoch budget; no early stopping",
    "versions": VERSIONS, "inputs": INPUTS}, RUN_DIR / "experiment_config.json")


## 2. Prepare shared article inputs and fixed histories

Reuse the initial/improved pipeline's preprocessing functions. Transformations are fitted only on training requests and histories. Catalogue metadata and frozen vectors use the shared ordered union; article publication time controls eligibility. Unshown articles are not negative labels. Test catalogue metadata may be used, but test requests and labels are not loaded.

In [ ]:
FRAMES, OFFSETS, TABLE_INFO, SPLIT_START = {}, {}, {}, {}
for split in ACTIVE_SPLITS:
    frame, info = D.load_candidate_table(PATHS[f"features_{split}"], split)
    offsets = info.pop("offsets")
    SPLIT_START[split] = pd.Timestamp(frame["impression_time"].min())
    if PATHS.get(f"behaviors_{split}"):
        check = D.cross_check_behaviours(frame, PATHS[f"behaviors_{split}"], split)
        assert check["feature_rows_not_in_behaviours"] == 0, check
        assert check["behaviour_candidates_not_in_features"] == 0, check
        assert check.get("label_mismatches", 0) == 0, check
        assert info["impressions"] == check["behaviour_rows"], "Missing original requests"
        info["behaviour_cross_check"] = check
    FRAMES[split], OFFSETS[split], TABLE_INFO[split] = frame, offsets, info
DEVICE_COLUMNS = TABLE_INFO["train"]["device_columns"]
assert all(TABLE_INFO[s]["device_columns"] == DEVICE_COLUMNS for s in ACTIVE_SPLITS)
log("All recorded requests/candidates:",
    {s: (TABLE_INFO[s]["impressions"], len(FRAMES[s])) for s in ACTIVE_SPLITS})


In [ ]:
VOCAB = json.loads(PATHS["feature_vocabulary"].read_text())
CATALOGUE, CATALOGUE_META = D.load_catalogue([PATHS["articles_train_val"], PATHS["articles_test"]])
CATALOGUE_INFO = CATALOGUE_META["info"]   # ordered union: train_val then test, first record wins
SNAPSHOT_IDS = {"train_val": CATALOGUE_META["snapshot_ids"][str(PATHS["articles_train_val"])],
                "test": CATALOGUE_META["snapshot_ids"][str(PATHS["articles_test"])]}
log("Catalogue:", CATALOGUE_INFO)

def expected_dim(kind, default):
    for key in ("embedding_metadata_train_val", "embedding_metadata_test"):
        if PATHS.get(key):
            meta = json.loads(PATHS[key].read_text())
            value = meta.get(f"{kind}_dimension")
            if value and value != default:
                raise ValueError(f"{key}: {kind} dimension {value} != {default}")
    return default
TEXT_DIM, IMAGE_DIM = expected_dim("text", 768), expected_dim("image", 1024)

VECTOR_INFO, VECTOR_SOURCES = {}, {"text": [], "image": []}
for kind, column, flag, dim in (("text", "text_embedding", "has_text_embedding", TEXT_DIM),
                                ("image", "image_embedding", "has_image_embedding", IMAGE_DIM)):
    for scope in ("train_val", "test"):
        ids, matrix, available, info = D.decode_vectors(PATHS[f"{kind}_{scope}"], column, flag, dim)
        VECTOR_SOURCES[kind].append((ids, matrix, available))
        VECTOR_INFO[f"{kind}_{scope}"] = info
        log(f"{kind:5s} {scope:9s}", {k: v for k, v in info.items() if k != "path"})
TEXT_LOOKUP, text_union = D.union_vectors(VECTOR_SOURCES["text"])
IMAGE_LOOKUP, image_union = D.union_vectors(VECTOR_SOURCES["image"])
for name, union in (("text", text_union), ("image", image_union)):
    VECTOR_INFO[f"{name}_union"] = {k: v for k, v in union.items() if k != "known_ids"}
    log(name, "union:", VECTOR_INFO[f"{name}_union"])
del VECTOR_SOURCES


In [ ]:
HISTORIES = {}
for split in ACTIVE_SPLITS:
    HISTORIES[split] = D.build_history(
        PATHS[f"history_{split}"], SPLIT_START[split], HISTORY_MAX_LEN,
        users_needed=FRAMES[split]["user_id"].to_numpy())
    log(split, "history:", HISTORIES[split].stats)


In [ ]:
def referenced_ids(split):
    hist = HISTORIES[split]
    mask = np.arange(hist.article_ids.shape[1])[None, :] < hist.lengths[:, None]
    return np.concatenate([FRAMES[split]["candidate_article_id"].to_numpy(), hist.article_ids[mask]])

all_ids = np.concatenate([referenced_ids(s) for s in ACTIVE_SPLITS] + [CATALOGUE.index.to_numpy()])
train_ids = referenced_ids("train")
article_stats = D.fit_article_stats(CATALOGUE, train_ids)
ID_VOCAB = {}  # learned article IDs are off; metadata/vectors support unseen articles
ARTICLES, ARTICLE_INFO = D.build_article_table(
    CATALOGUE, all_ids, VOCAB, article_stats, TEXT_LOOKUP, IMAGE_LOOKUP,
    TEXT_DIM, IMAGE_DIM, 8, ID_VOCAB)
PREPROCESSING = {
    "numeric_stats": D.fit_numeric_stats(FRAMES["train"], HISTORIES["train"], OFFSETS["train"], ARTICLES),
    "article_stats": article_stats, "history_max_len": HISTORY_MAX_LEN,
    "device_columns": DEVICE_COLUMNS, "feature_vocabulary": VOCAB,
    "text_dim": TEXT_DIM, "image_dim": IMAGE_DIM, "max_topics": 8,
    "candidate_features": D.CANDIDATE_FEATURE_NAMES,
}
ARRAYS = {s: D.build_split_arrays(s, FRAMES[s], OFFSETS[s], HISTORIES[s], ARTICLES,
                                PREPROCESSING["numeric_stats"], DEVICE_COLUMNS) for s in ACTIVE_SPLITS}
coverage = {s: D.coverage_report(ARRAYS[s], ARTICLES) for s in ACTIVE_SPLITS}
for a in ARRAYS.values():
    assert np.isfinite(a.imp_context).all() and np.isfinite(a.cand_features).all()
    assert np.isfinite(a.hist_features).all()
assert ARRAYS["train"].event_names[M.GAP_PREV_FEATURE] == "log_gap_prev_h_z"
assert ARRAYS["train"].event_names[M.GAP_PREV_MISSING] == "gap_prev_missing"
log("Article inputs:", ARTICLE_INFO)
log("Feature order:", {"context": ARRAYS["train"].context_names,
                        "candidate": ARRAYS["train"].candidate_names,
                        "event": ARRAYS["train"].event_names,
                        "article_numeric": ARTICLES.numeric_names})


## 3. Fixed selection and separate reporting requests

In [ ]:
UNION_IDS = CATALOGUE.index.to_numpy(dtype=np.int64)
POLICY = D.EligibilityPolicy(["train_val/articles.parquet", "test/articles.parquet"])
INDEX = D.build_catalogue_index("all", "union", UNION_IDS, ARTICLES)
ELIGIBILITY = {s: D.eligibility_summary(ARRAYS[s], INDEX) for s in ACTIVE_SPLITS}
CLICK_ELIGIBILITY = {s: D.clicked_eligibility(ARRAYS[s], INDEX, ARTICLES) for s in ACTIVE_SPLITS}
VAL = ARRAYS["validation"]
SELECTION_SAMPLE, selection_bids = D.select_requests(
    VAL.user_id, VAL.impression_id, VAL.imp_time_us, "validation", SEED, SELECTION_REQUESTS)
SELECTION = {
    "objective": SELECTION_OBJECTIVE, "tiebreak": SELECTION_TIEBREAK,
    "earlier_checkpoint_on_equal_metrics": True, "interval": "every epoch",
    "selection_k": SELECTION_K, "pool_policy": POLICY.to_json(),
    "freshness_grid": list(FRESHNESS_GRID), "within_epoch_tie": "smaller freshness weight",
    "reserved_final_partition": "labelled project test; outcomes deferred until full pipeline freeze",
    "sampling": "smallest BLAKE2b-64(seed:split:behaviour_id), then behaviour_id; chronological output",
    "seed": SEED, "requests": selection_bids, "population_requests": VAL.n_impressions,
    "logged_metrics": [SELECTION_OBJECTIVE, SELECTION_TIEBREAK,
                       "eligible_only_recall@100", "diagnostic_recorded_slate_auc", "freshness_weight"],
}
(RUN_DIR / "selection").mkdir()
D.write_json(SELECTION, RUN_DIR / "selection" / "declaration.json")
pd.DataFrame({"behaviour_id": selection_bids}).to_parquet(
    RUN_DIR / "selection" / "validation_requests.parquet", index=False)
CATALOGUE_FINGERPRINT = hashlib.sha256(UNION_IDS.astype("<i8").tobytes()).hexdigest()
log("Complete pools and scoring work:", ELIGIBILITY)

remaining = np.setdiff1d(np.arange(VAL.n_impressions), SELECTION_SAMPLE)
if len(remaining) < REPORT_REQUESTS:
    raise ValueError("Not enough non-selection validation requests for the reporting sample.")
report_positions, report_bids = D.select_requests(
    VAL.user_id[remaining], VAL.impression_id[remaining], VAL.imp_time_us[remaining],
    "validation", SEED, REPORT_REQUESTS)
REPORT_SAMPLE = remaining[report_positions]
assert len(SELECTION_SAMPLE) == SELECTION_REQUESTS
assert len(REPORT_SAMPLE) == REPORT_REQUESTS
assert not np.intersect1d(SELECTION_SAMPLE, REPORT_SAMPLE).size
pd.DataFrame({"behaviour_id": report_bids, "request_index": REPORT_SAMPLE}).to_parquet(
    RUN_DIR / "selection" / "reporting_requests.parquet", index=False)
SELECTION["checkpoint_objective"] = CHECKPOINT_OBJECTIVE
D.write_json(SELECTION, RUN_DIR / "selection" / "declaration.json")
copy_tree(RUN_DIR / "selection", PERSIST_RUN_DIR / "selection")
log("Selection requests:", len(SELECTION_SAMPLE), "| Reporting requests:", len(REPORT_SAMPLE))


In [ ]:
ARTICLE_TENSORS = M.article_table_tensors(ARTICLES, DEVICE)
TENSORS = {s: M.SplitTensors(ARRAYS[s], DEVICE) for s in ACTIVE_SPLITS}
VAL_CLICK_LOOKUP = D.clicks_by_request(CLICK_ELIGIBILITY["validation"])
selection_slates = D.subset_split_arrays(VAL, SELECTION_SAMPLE)
SELECTION_TENSORS = M.SplitTensors(selection_slates, DEVICE)

def model_config(overrides=None):
    sample = ARRAYS["train"]
    gap = PREPROCESSING["numeric_stats"]["log_gap_to_impression_h"]
    params = {**MODEL_HPARAMS, **(overrides or {})}
    return M.ModelConfig(
        variant="attention", n_categories=len(VOCAB["category"]) + 3,
        n_types=len(VOCAB["article_type"]) + 3, n_topics=len(VOCAB["topics"]) + 2,
        n_article_numeric=ARTICLES.numeric.shape[1], text_dim_in=TEXT_DIM, image_dim_in=IMAGE_DIM,
        n_id_buckets=1, n_event_features=sample.hist_features.shape[-1],
        n_candidate_features=sample.cand_features.shape[1], n_context_features=sample.imp_context.shape[1],
        gap_mean=gap["mean"], gap_std=gap["std"], **params)


## 4. Shared evaluation and preflight checks

One catalogue-scoring pass evaluates raw neural scores and all requested freshness weights. Each arm selects its own weight using selection requests only. Reporting always includes the selected checkpoint with weight zero as well as its selected freshness weight; the raw row is not an independently selected raw checkpoint when `CHECKPOINT_OBJECTIVE="blended"`.

The preflight checks a tiny forward/backward pass at each depth and compares original and batched inference for both architectures. It runs before the expensive fits.

In [ ]:
def configure_model(depth):
    cfg = model_config({"lstm_layers": depth, "lstm_hidden": 128})
    net = M.build_model(cfg, ARTICLE_TENSORS, DEVICE)
    if depth > 1 and INTER_LAYER_DROPOUT is not None:
        net.lstm.dropout = float(INTER_LAYER_DROPOUT)
    return net, cfg


def catalogue_eval(net, requests, weights):
    net.eval()
    weights = tuple(sorted(set(float(w) for w in weights)))
    torch.cuda.synchronize()
    started = time.perf_counter()
    scorer = F.BatchedCatalogueScorer(net, TENSORS["validation"], INDEX,
        PREPROCESSING["numeric_stats"]["log_candidate_age_h"],
        PAIR_CHUNK, REQUEST_BATCH, requests=requests)
    torch.cuda.synchronize()
    setup_seconds = time.perf_counter() - started
    started = time.perf_counter()
    hits = {w: {} for w in weights}
    for block, counts, outputs in scorer.batches(requests, EXPORT_K, weights):
        for weight, output in outputs.items():
            for row, i in enumerate(block):
                ids = output["article_ids"][row, :min(EXPORT_K, int(counts[row]))]
                hit = D.click_hits(VAL_CLICK_LOOKUP, int(i), ids, (100, 200))
                if hit is not None:
                    hits[weight][int(i)] = hit
    torch.cuda.synchronize()
    score_seconds = time.perf_counter() - started
    metrics = {w: D.recall_from_hits(h, CLICK_ELIGIBILITY["validation"],
        (100, 200), requests_scored=len(requests)) for w, h in hits.items()}
    return metrics, hits, {"cache_setup_seconds": setup_seconds,
        "scoring_and_metrics_seconds": score_seconds,
        "milliseconds_per_request_including_setup":
            1000 * (setup_seconds + score_seconds) / len(requests)}


def choose_weight(metrics):
    # The smaller weight wins ties, matching the improved notebook.
    return max(metrics, key=lambda w: (metrics[w][SELECTION_OBJECTIVE],
        metrics[w][SELECTION_TIEBREAK], -w))


preflight = {}
for depth in DEPTHS:
    M.set_seed(SEED)
    net, cfg = configure_model(depth)
    batch = TENSORS["train"].batch(torch.arange(2, device=DEVICE))
    net.train()
    logits = net(batch)
    assert logits.shape == batch["labels"].shape and torch.isfinite(logits).all()
    loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, batch["labels"])
    loss.backward()
    grad = net.lstm.weight_ih_l0.grad
    assert grad is not None and torch.isfinite(grad).all()
    if depth == 2:
        grad = net.lstm.weight_ih_l1.grad
        assert grad is not None and torch.isfinite(grad).all()
    net.zero_grad(set_to_none=True)
    net.eval()
    requests = SELECTION_SAMPLE[:4]
    reference = M.CatalogueScorer(net, TENSORS["validation"], INDEX,
        PREPROCESSING["numeric_stats"]["log_candidate_age_h"], requests=requests)
    batched = F.BatchedCatalogueScorer(net, TENSORS["validation"], INDEX,
        PREPROCESSING["numeric_stats"]["log_candidate_age_h"],
        PAIR_CHUNK, REQUEST_BATCH, requests=requests)
    check = F.benchmark_equivalence(reference, batched, requests, RELOAD_TOLERANCE)
    assert check["passed"], check
    preflight[str(depth)] = {"parameters": M.count_parameters(net),
        "actual_inter_layer_dropout": net.lstm.dropout, "inference_check": check}
    del net, reference, batched, logits, loss, batch, grad
    torch.cuda.empty_cache()
D.write_json(preflight, RUN_DIR / "preflight.json")
copy_tree(RUN_DIR, PERSIST_RUN_DIR)
log("Preflight checks passed:", preflight)


## 5. Persist shared inference inputs

Article inputs, mappings, fitted preprocessing and helper sources are stored once for all arms. Each arm stores its own selected weights, architecture, actual inter-layer dropout, freshness recipe, histories and report. A local checkpoint is copied to Drive after each completed epoch. This notebook starts a new experiment on rerun; it does not promise automatic interrupted-run resumption.

In [ ]:
SHARED_DIR = RUN_DIR / "shared"
(SHARED_DIR / "code").mkdir(parents=True)
np.savez_compressed(SHARED_DIR / "article_inputs.npz",
    **{f.name: np.asarray(getattr(ARTICLES, f.name)) for f in fields(D.ArticleTable)})
np.savez_compressed(SHARED_DIR / "mappings.npz", article_table_ids=ARTICLES.article_ids,
    catalogue_source_ids=UNION_IDS)
D.write_json(PREPROCESSING, SHARED_DIR / "preprocessing.json")
D.write_json({"request_context": ARRAYS["train"].context_names,
    "candidate": ARRAYS["train"].candidate_names,
    "history_event": ARRAYS["train"].event_names,
    "article_numeric": ARTICLES.numeric_names}, SHARED_DIR / "feature_dictionary.json")
D.write_json(POLICY.to_json(), SHARED_DIR / "catalogue_policy.json")
for name in ("lstm_data.py", "lstm_model.py", "lstm_inference.py"):
    shutil.copy2(CODE_DIR / name, SHARED_DIR / "code" / name)
(RUN_DIR / "requirements.txt").write_text(
    "\n".join(f"{k}=={v}" for k, v in VERSIONS.items()) + "\n")
copy_tree(RUN_DIR, PERSIST_RUN_DIR)


## 6. Train matching one- and two-layer models

Fresh weights for every seed/depth. Hyperparameters are inherited from the initial one-layer tuning, not independently tuned optima for each depth. Both depths train on all training requests for the same fixed epoch budget. Parameter initialisation and dropout consume different random streams across architectures even with matching seeds; matching seeds do not make every stochastic operation identical.

`CHECKPOINT_OBJECTIVE="blended"` selects by best freshness-adjusted Recall@100. `"raw"` selects by weight-zero Recall@100, while still recording the other weights. Smaller weights break within-epoch ties and earlier checkpoints win equal metrics. No model is selected using reporting results.

In [ ]:
RESULTS = []
for seed in SEEDS:
    for depth in DEPTHS:
        arm = f"layers{depth}_seed{seed}"
        arm_dir = RUN_DIR / "arms" / arm
        arm_dir.mkdir(parents=True, exist_ok=False)
        persistent_arm = PERSIST_RUN_DIR / "arms" / arm
        M.set_seed(seed)
        model, cfg = configure_model(depth)
        training_cfg = M.TrainConfig(**{**TRAIN_HPARAMS, "seed": seed,
            "max_epochs": MAX_EPOCHS, "early_stopping_patience": MAX_EPOCHS + 1,
            "lr_patience": MAX_EPOCHS + 1, "max_train_impressions_per_epoch": None})
        curves = []

        def select_validation(net):
            metrics, _, timing = catalogue_eval(net, SELECTION_SAMPLE, FRESHNESS_GRID)
            best_weight = choose_weight(metrics)
            selected_weight = best_weight if CHECKPOINT_OBJECTIVE == "blended" else 0.0
            chosen = {**metrics[selected_weight], "freshness_weight": selected_weight}
            curves.append({"epoch": len(curves) + 1, "best_blend_weight": best_weight,
                "checkpoint_weight": selected_weight, "timing": timing,
                "curve": [{"freshness_weight": w, **m} for w, m in metrics.items()]})
            D.write_json(curves, arm_dir / "selection_curves.json")
            return chosen

        def persist_epoch(epoch, row):
            copy_tree(arm_dir, persistent_arm)

        arm_selection = {**SELECTION, "depth": depth, "model_seed": seed}
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        started = time.perf_counter()
        history, best = M.train_model(model, TENSORS["train"], select_validation,
            arm_selection, training_cfg, arm_dir / "training", resume=False,
            on_epoch_end=persist_epoch, log=log)
        torch.cuda.synchronize()
        fit_seconds = time.perf_counter() - started
        peak_bytes = torch.cuda.max_memory_allocated()
        assert len(history) == MAX_EPOCHS
        model.eval()
        selected_curve = curves[best["epoch"] - 1]
        weight = float(selected_curve["best_blend_weight"])
        M.atomic_torch_save(model.state_dict(), arm_dir / "model_state.pt")
        D.write_json(cfg.to_json(), arm_dir / "model_config.json")
        D.write_json({"shared_inputs": "../../shared", "depth": depth, "seed": seed,
            "actual_inter_layer_dropout": model.lstm.dropout,
            "train_config": asdict(training_cfg), "checkpoint_objective": CHECKPOINT_OBJECTIVE,
            "selected_epoch": best["epoch"], "parameters": M.count_parameters(model)},
            arm_dir / "arm_config.json")
        D.write_json({"formula": "neural_logit - freshness_weight * log1p(age_hours)",
            "freshness_weight": weight, "age_convention": "same as 2.1.3 and lstm_inference.py",
            "score_semantics": "uncalibrated ranking value", "selected_epoch": best["epoch"]},
            arm_dir / "score_recipe.json")
        copy_tree(arm_dir, persistent_arm)  # Persist before reporting.
        metrics, hits, timing = catalogue_eval(model, REPORT_SAMPLE, (0., weight))
        rows = []
        for score_type, w in (("raw", 0.), ("blended", weight)):
            result = {"arm": arm, "depth": depth, "seed": seed, "score_type": score_type,
                "freshness_weight": w, "selected_epoch": best["epoch"],
                "fit_and_selection_seconds": fit_seconds, "peak_allocated_gpu_bytes": peak_bytes,
                "trainable_parameters": M.count_parameters(model)["trainable_parameters"],
                **timing, **{k: v for k, v in metrics[w].items() if k != "split_click_eligibility"}}
            RESULTS.append(result)
            for i, h in hits[w].items():
                rows.append({"arm": arm, "seed": seed, "depth": depth,
                    "score_type": score_type, "request_index": i, "user_id": int(VAL.user_id[i]),
                    "recall100": h["hits@100"] / h["n"],
                    "ndcg100": h["dcg@100"] / h["idcg@100"],
                    "recall200": h["hits@200"] / h["n"]})
        pd.DataFrame(rows).to_parquet(arm_dir / "report_per_request.parquet", index=False)
        D.write_json([r for r in RESULTS if r["arm"] == arm], arm_dir / "report_metrics.json")
        pd.DataFrame(history).to_csv(arm_dir / "training_history.csv", index=False)

        # Check persisted weights using a small recorded-candidate batch.
        expected = M.predict_logits(model, SELECTION_TENSORS, batch_size=256)[:128]
        reloaded = M.build_model(M.ModelConfig.from_json(json.loads(
            (persistent_arm / "model_config.json").read_text())), ARTICLE_TENSORS, DEVICE)
        reloaded.load_state_dict(torch.load(persistent_arm / "model_state.pt",
            map_location=DEVICE, weights_only=True))
        reloaded.eval()
        actual = M.predict_logits(reloaded, SELECTION_TENSORS, batch_size=256)[:128]
        difference = float(np.max(np.abs(expected - actual)))
        assert np.allclose(expected, actual, atol=RELOAD_TOLERANCE, rtol=0), difference
        D.write_json({"passed": True, "pairs": len(expected), "max_abs_difference": difference,
            "scope": "persisted weights/config with in-memory shared article inputs; evaluation mode",
            "tolerance": RELOAD_TOLERANCE}, arm_dir / "reload_check.json")
        copy_tree(arm_dir, persistent_arm)
        pd.DataFrame(RESULTS).to_csv(RUN_DIR / "results.csv", index=False)
        D._copy_atomic(RUN_DIR / "results.csv", PERSIST_RUN_DIR / "results.csv")
        log("Completed", arm, "| selected epoch", best["epoch"], "| blend weight", weight)
        del model, reloaded
        torch.cuda.empty_cache()


## 7. Paired comparison and interpretation

Report two-layer minus one-layer differences on identical reporting requests and matching seeds. Average seed-matched per-request differences before resampling users, retaining each user's requests together. These 95% intervals describe user-sampling uncertainty conditional on the fitted models; they do not fully represent training uncertainty. With one seed, conclusions are exploratory.

The raw and blended rows share a selected checkpoint. Under blended checkpoint selection, the raw row is a diagnostic, not a separately optimised raw model. Timing includes Python metric bookkeeping and cache setup, and the same pass may evaluate two weights. It is not a standalone production inference benchmark. Compare quality with parameter count, allocated GPU memory and training cost. No results are prefilled, and test outcomes remain unused.

In [ ]:
results = pd.DataFrame(RESULTS)
display(results[["depth", "seed", "score_type", "selected_epoch", "freshness_weight",
    "known_click_recall@100", "observed_click_ndcg@100", "known_click_recall@200",
    "trainable_parameters", "fit_and_selection_seconds"]])
summary = results.groupby(["depth", "score_type"], as_index=False).agg(
    seeds=("seed", "count"), recall100=("known_click_recall@100", "mean"),
    ndcg100=("observed_click_ndcg@100", "mean"),
    recall200=("known_click_recall@200", "mean"),
    fit_seconds=("fit_and_selection_seconds", "mean"))
summary.to_csv(RUN_DIR / "summary.csv", index=False)
display(summary)

pairs = []
for seed in SEEDS:
    frames = [pd.read_parquet(RUN_DIR / "arms" / f"layers{d}_seed{seed}" /
        "report_per_request.parquet") for d in DEPTHS]
    joined = frames[0].merge(frames[1], on=["request_index", "user_id", "seed", "score_type"],
        suffixes=("_one", "_two"), validate="one_to_one")
    assert len(joined) == len(frames[0]) == len(frames[1])
    for metric in ("recall100", "ndcg100", "recall200"):
        for score_type, part in joined.groupby("score_type"):
            for row in part.to_dict("records"):
                pairs.append({"seed": seed, "request_index": row["request_index"],
                    "user_id": row["user_id"], "score_type": score_type, "metric": metric,
                    "delta": row[f"{metric}_two"] - row[f"{metric}_one"]})
paired = pd.DataFrame(pairs)
assert paired.groupby(["request_index", "score_type", "metric"]).seed.nunique().eq(len(SEEDS)).all()
averaged = paired.groupby(["request_index", "user_id", "score_type", "metric"], as_index=False).delta.mean()
intervals = []
rng = np.random.default_rng(SEED)
for (score_type, metric), part in averaged.groupby(["score_type", "metric"]):
    users = part.groupby("user_id").delta.agg(["sum", "count"])
    sums, counts = users["sum"].to_numpy(), users["count"].to_numpy()
    draws = np.empty(2000)
    for b in range(len(draws)):
        selected = rng.integers(0, len(users), size=len(users))
        draws[b] = sums[selected].sum() / counts[selected].sum()
    low, high = np.quantile(draws, [0.025, 0.975])
    intervals.append({"score_type": score_type, "metric": metric,
        "two_minus_one": float(part.delta.mean()), "ci95_low": float(low),
        "ci95_high": float(high), "seeds": len(SEEDS), "users": len(users)})
comparison = pd.DataFrame(intervals)
comparison.to_csv(RUN_DIR / "paired_comparison.csv", index=False)
averaged.to_parquet(RUN_DIR / "paired_request_differences.parquet", index=False)
display(comparison)
copy_tree(RUN_DIR, PERSIST_RUN_DIR)
log("Saved experiment:", PERSIST_RUN_DIR)
log("No project-test outcomes evaluated; no train/test ranking exports generated.")
